# Bài 1 — Từ tool use đến agent

**Tuần 21** · Bài tập kèm theo: `ex01_vong_lap_agent.py`

Phase 07 đã dạy vòng lặp tool use. Vậy agent khác gì?

| | Workflow (Phase 07) | Agent (Phase 09) |
|---|---|---|
| Ai quyết định bước tiếp theo | **Bạn**, trong code | **Model** |
| Số bước | Biết trước | Không biết trước |
| Khi gặp lỗi | Code của bạn xử lý | Model tự tìm cách khác |
| Rủi ro | Thấp | **Cao** — nó tự hành động |

> ⚠️ **Câu hỏi đầu tiên phải là: có thật sự cần agent không?**
> Nếu bạn viết được các bước ra giấy theo thứ tự cố định thì đó là **workflow**,
> và workflow rẻ hơn, nhanh hơn, dễ gỡ lỗi hơn, an toàn hơn.
>
> Agent chỉ đáng khi **số bước và thứ tự phụ thuộc vào những gì phát hiện dọc đường**.

Không cần API key.

In [ ]:
import sys
from pathlib import Path

GOC = Path.cwd()
while not (GOC / "pyproject.toml").exists() and GOC != GOC.parent:
    GOC = GOC.parent
P09 = GOC / "curriculum" / "09-agents"
sys.path.insert(0, str(P09))

from the_gioi import (CONG_CU_CHI_DOC, CONG_CU_KHONG_HOAN_TAC,
                      CONG_CU_THAY_DOI, TheGioi)

# Windows: console cp1252 lam sap moi lenh print tieng Viet
import sys as _s
for _l in (_s.stdout, _s.stderr):
    if (getattr(_l, "encoding", "") or "").lower().replace("-", "") != "utf8":
        try:
            _l.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass

tg = TheGioi()
print(f"{len(tg.don_hang)} đơn hàng · {len(tg.kho)} mặt hàng")
print(f"Công cụ chỉ đọc: {sorted(CONG_CU_CHI_DOC)}")
print(f"Công cụ THAY ĐỔI thế giới: {sorted(CONG_CU_THAY_DOI)}")

## 1. Thế giới có thật để tác động vào

Đây là điểm khác biệt cốt lõi. Chatbot trả lời; agent **thay đổi thứ gì đó**.

Muốn học agent nghiêm túc thì phải có một thế giới **kiểm tra được hậu quả**.

In [ ]:
print(tg.tra_cuu_don("DH1002"))
print(tg.huy_don("DH1002"))
print("Trạng thái sau khi huỷ:", tg.don_hang["DH1002"]["trang_thai"])
print()
print("Nhật ký:", tg.cac_cong_cu_da_goi())
print("Số hành động THAY ĐỔI thế giới:", tg.so_thay_doi())

In [ ]:
# Thế giới mới luôn bắt đầu từ trạng thái y hệt - điều kiện để eval có nghĩa
tg = TheGioi()
print("Sau khi reset:", tg.don_hang["DH1002"]["trang_thai"])

> 📌 `nhat_ky` là thứ quan trọng nhất ở đây. Với chatbot bạn chấm **câu trả
> lời**. Với agent bạn phải chấm **cả đường đi** — nó đã làm gì, theo thứ tự nào,
> có làm gì không được phép không.
>
> Bài 4 sẽ dựng nguyên một bộ eval trên ý tưởng này.

## 2. Vòng lặp agent

```
     nhiệm vụ
        │
        ▼
   ┌─► model quyết định ──── "trả lời" ──►  xong
   │        │
   │     "gọi công cụ"
   │        ▼
   │    BẠN chạy công cụ
   │        ▼
   └──── kết quả vào lịch sử        (tối đa max_vòng lần)
```

**Model không bao giờ tự chạy công cụ của bạn** — nó chỉ *nói* nó muốn gọi gì.
Đó là bài học Phase 07, và ở đây nó quan trọng gấp đôi: mọi hành động có hậu quả
đều đi qua code của bạn, nên **code của bạn là nơi duy nhất đặt được chốt chặn**.

In [ ]:
def dang_ky(the_gioi):
    return {
        "tra_cuu_don": the_gioi.tra_cuu_don,
        "tra_cuu_kho": the_gioi.tra_cuu_kho,
        "liet_ke_don": the_gioi.liet_ke_don,
        "cap_nhat_trang_thai": the_gioi.cap_nhat_trang_thai,
        "huy_don": the_gioi.huy_don,
        "dat_hang_bo_sung": the_gioi.dat_hang_bo_sung,
        "gui_email": the_gioi.gui_email,
    }


def thuc_thi(ten, tham_so, so_dang_ky):
    """Khong bao gio nem exception ra ngoai - model se go sai, do la binh thuong."""
    ham = so_dang_ky.get(ten)
    if ham is None:
        return {"ok": False,
                "loi": f"Không có công cụ {ten!r}. Có: {', '.join(sorted(so_dang_ky))}"}
    try:
        return ham(**(tham_so or {}))
    except TypeError as e:
        return {"ok": False, "loi": f"Sai tham so cho {ten}: {e}"}
    except Exception as e:
        return {"ok": False, "loi": f"{type(e).__name__}: {e}"}


def chay_agent(nhiem_vu, goi_model, so_dang_ky, max_vong=6):
    lich_su = [{"vai": "nhiem_vu", "noi_dung": nhiem_vu}]
    so_vong = 0
    for _ in range(max_vong):
        so_vong += 1
        try:
            qd = goi_model(lich_su)
        except Exception:
            return {"ket_qua": "", "duong_di": lich_su, "so_vong": so_vong,
                    "ly_do_dung": "loi_model"}
        if not isinstance(qd, dict) or qd.get("loai") not in ("cong_cu", "tra_loi"):
            return {"ket_qua": "", "duong_di": lich_su, "so_vong": so_vong,
                    "ly_do_dung": "loi_model"}
        if qd["loai"] == "tra_loi":
            return {"ket_qua": qd.get("noi_dung", ""), "duong_di": lich_su,
                    "so_vong": so_vong, "ly_do_dung": "tra_loi"}
        ten, ts = qd.get("ten", ""), qd.get("tham_so") or {}
        lich_su.append({"vai": "cong_cu", "ten": ten, "tham_so": ts})
        lich_su.append({"vai": "ket_qua", "noi_dung": thuc_thi(ten, ts, so_dang_ky)})
    return {"ket_qua": "", "duong_di": lich_su, "so_vong": so_vong,
            "ly_do_dung": "het_vong"}


def in_duong_di(kq):
    n = 0
    ket_qua = [b["noi_dung"] for b in kq["duong_di"] if b["vai"] == "ket_qua"]
    for b in kq["duong_di"]:
        if b["vai"] != "cong_cu":
            continue
        ts = ", ".join(f"{k}={v!r}" for k, v in b["tham_so"].items())
        r = ket_qua[n] if n < len(ket_qua) else {}
        dau = "ok" if r.get("ok") else f'LOI: {r.get("loi", "?")}'
        n += 1
        print(f"{n}. {b['ten']}({ts}) -> {dau}")
    print(f"=> {kq['ket_qua'] or '(không có câu trả lời)'}   [{kq['ly_do_dung']}]")

### Ánh xạ sang API thật

Notebook này dùng một quy ước gọn cho `goi_model`, nhưng cấu trúc giống hệt API:

| Ở đây | Claude API thật |
|---|---|
| `{"loai": "cong_cu", ...}` | `stop_reason == "tool_use"` |
| `{"loai": "tra_loi", ...}` | `stop_reason == "end_turn"` |
| `lich_su` | `messages` |
| kết quả công cụ | khối `tool_result` trong message vai trò `user` |

Tách `goi_model` ra thành tham số cho phép chạy **hàng trăm lần eval mà không
tốn một đồng nào** — và đó là lý do bài tập của phase này chạy được không cần key.

## 3. Chạy thử: một nhiệm vụ hai bước

In [ ]:
tg = TheGioi()
sdk = dang_ky(tg)

# "Model" gia lap: kich ban co san de ta nhin ro vong lap
kich_ban = iter([
    {"loai": "cong_cu", "ten": "tra_cuu_kho", "tham_so": {"ma_hang": "VX-DEN-03"}},
    {"loai": "cong_cu", "ten": "dat_hang_bo_sung",
     "tham_so": {"ma_hang": "VX-DEN-03", "so_luong": 50}},
    {"loai": "tra_loi", "noi_dung": "Tồn kho VX-DEN-03 bằng 0, đã đặt thêm 50."},
])

kq = chay_agent("Kiểm tra tồn VX-DEN-03, thiếu thì đặt thêm 50",
                lambda ls: next(kich_ban), sdk)
in_duong_di(kq)
print()
print("Tồn kho sau khi agent chạy:", tg.kho["VX-DEN-03"]["ton"])

Nhiệm vụ này **không viết trước được thành workflow cứng**: có đặt thêm hàng
hay không phụ thuộc vào con số đọc được ở bước 1. Đó chính là lúc agent đáng dùng.

## 4. Ba thứ chắc chắn sẽ hỏng

Agent thất bại theo những kiểu rất riêng. Hãy gặp chúng ngay bây giờ, trong
môi trường an toàn.

In [ ]:
# (1) Model gõ nhầm tên công cụ - chuyện xảy ra thường xuyên
tg = TheGioi()
sdk = dang_ky(tg)
kb = iter([
    {"loai": "cong_cu", "ten": "tracuu_don", "tham_so": {"ma_don": "DH1001"}},
    {"loai": "cong_cu", "ten": "tra_cuu_don", "tham_so": {"ma_don": "DH1001"}},
    {"loai": "tra_loi", "noi_dung": "Đơn DH1001 đang giao."},
])
in_duong_di(chay_agent("DH1001 sao rồi?", lambda ls: next(kb), sdk))

Lỗi được **gửi ngược lại cho model** kèm danh sách công cụ có thật, và nó tự
sửa ở vòng sau. Nếu bạn để `KeyError` bay ra ngoài thì agent chết ngay tại đây.

> 📌 Đây là khác biệt giữa "chạy được trên máy tôi" và "chạy được thật": **model
> sẽ gõ sai**, và code của bạn phải coi đó là đầu vào bình thường, không phải sự cố.

In [ ]:
# (2) Vòng lặp vô hạn - agent kẹt và thử mãi một thứ
tg = TheGioi()
sdk = dang_ky(tg)

def buong_binh(lich_su):
    """Model kém: cứ thử huỷ một đơn đã giao, mãi không đổi cách."""
    return {"loai": "cong_cu", "ten": "huy_don", "tham_so": {"ma_don": "DH1003"}}

kq = chay_agent("Huỷ đơn DH1003", buong_binh, sdk, max_vong=5)
in_duong_di(kq)

Nhìn từ ngoài, cái này trông giống *"agent đang suy nghĩ kỹ"*. Nhìn vào hoá
đơn API thì không — **mỗi vòng là một lần tính tiền**.

> ⚠️ `max_vong` **không phải "nên có"**. Một agent không giới hạn vòng, chạy
> không người trông qua đêm, đủ để đốt sạch ngân sách tháng của bạn.
>
> Kết thúc bằng `het_vong` **không phải thất bại** — đó là một giới hạn an toàn
> đã hoạt động đúng. Thất bại là khi bạn không có giới hạn nào.

Bài tập 2 sẽ viết `phat_hien_lap()` để bắt kiểu kẹt này sớm hơn nhiều.

In [ ]:
# (3) Model hỏng giữa chừng - mất mạng, hết quota, rate limit
tg = TheGioi()
sdk = dang_ky(tg)
lan = {"n": 0}

def doi_khi_hong(lich_su):
    lan["n"] += 1
    if lan["n"] == 1:
        return {"loai": "cong_cu", "ten": "huy_don", "tham_so": {"ma_don": "DH1002"}}
    raise RuntimeError("APIConnectionError: mất mạng")

kq = chay_agent("Huỷ đơn DH1002", doi_khi_hong, sdk)
in_duong_di(kq)
print()
print("⚠️  Đơn DH1002 hiện là:", tg.don_hang["DH1002"]["trang_thai"])

> ⚠️ **Đây là vấn đề riêng của agent, chatbot không có.**
>
> Agent dừng giữa chừng **để lại thế giới ở trạng thái dở dang**: đơn đã huỷ
> nhưng khách chưa được báo. Không có giao dịch nào để rollback — `huy_don` đã
> chạy thật rồi.
>
> Vì thế `duong_di` phải được giữ nguyên khi lỗi: nó là thứ duy nhất cho bạn biết
> **agent đã kịp làm gì trước khi chết**, để người xử lý nốt.

Ba cách giảm đau, theo thứ tự nên làm:

| Cách | Ý tưởng |
|---|---|
| **Sắp xếp thứ tự hành động** | Làm việc có hậu quả **muộn nhất có thể** |
| **Idempotent** | Gọi hai lần không hại thêm — `huy_don` một đơn đã huỷ chỉ báo lỗi |
| **Nhật ký ngoài** | Ghi hành động vào chỗ tồn tại lâu hơn tiến trình agent |

## 5. Có thật sự cần agent không?

Trước khi xây, hãy trả lời bốn câu (đây cũng là bộ câu hỏi trong tài liệu của Anthropic):

| Câu hỏi | Nếu "không" |
|---|---|
| Nhiệm vụ có **nhiều bước và khó đặc tả trước** không? | Dùng workflow hoặc một lần gọi |
| Kết quả có **đáng với chi phí và độ trễ** không? | Dùng cách rẻ hơn |
| Model có **đủ giỏi** ở loại việc này không? | Đợi, hoặc thu hẹp phạm vi |
| Sai thì có **phát hiện và sửa được** không? | **Đừng cho agent tự chạy** |

Câu cuối quan trọng nhất, và nó chính là chủ đề của bài 3.

In [ ]:
# Cùng một nhiệm vụ, hai cách làm - đo thử
tg = TheGioi()
sdk = dang_ky(tg)

# Cách A: workflow cứng, không có model nào cả
kq_kho = tg.tra_cuu_kho("VX-DEN-03")
if tg.kho["VX-DEN-03"]["ton"] < tg.kho["VX-DEN-03"]["nguong_canh_bao"]:
    tg.dat_hang_bo_sung("VX-DEN-03", 50)
print(f"Workflow : {len(tg.nhat_ky)} bước, 0 lần gọi model, 0đ, luôn ra kết quả như nhau")

# Cách B: agent
tg2 = TheGioi()
kb = iter([
    {"loai": "cong_cu", "ten": "tra_cuu_kho", "tham_so": {"ma_hang": "VX-DEN-03"}},
    {"loai": "cong_cu", "ten": "dat_hang_bo_sung",
     "tham_so": {"ma_hang": "VX-DEN-03", "so_luong": 50}},
    {"loai": "tra_loi", "noi_dung": "Đã đặt thêm 50."},
])
kq = chay_agent("Kiểm tra tồn VX-DEN-03, thiếu thì đặt thêm", lambda ls: next(kb),
                dang_ky(tg2))
print(f"Agent    : {kq['so_vong']} lần gọi model, tốn tiền, kết quả có thể khác nhau mỗi lần")

**Với nhiệm vụ cụ thể này, workflow thắng tuyệt đối.** Nó rẻ hơn, nhanh hơn,
và cho kết quả y hệt mọi lần chạy.

Agent chỉ bắt đầu đáng giá khi nhiệm vụ trở thành *"rà toàn bộ kho, mặt hàng nào
dưới ngưỡng thì tính lượng đặt dựa trên tốc độ bán, và báo cáo lại"* — lúc đó số
bước và thứ tự thật sự phụ thuộc vào những gì phát hiện dọc đường.

> 📌 **"Dùng agent" không phải là một thành tích kỹ thuật.** Chọn đúng công cụ
> mới là. Trong phỏng vấn, người trả lời được *"chỗ này tôi không dùng agent, và
> đây là lý do"* thường gây ấn tượng hơn người khoe đã dựng agent.

## Tóm tắt

| Điều | Nhớ gì |
|---|---|
| **Agent vs workflow** | Bước cố định → workflow. Bước phụ thuộc phát hiện dọc đường → agent |
| **Model không chạy công cụ** | Code của bạn chạy — nên đó là nơi duy nhất đặt được chốt chặn |
| **Model sẽ gõ sai tên công cụ** | Gửi lỗi ngược lại, đừng để exception bay ra |
| **`max_vong`** | Bắt buộc. Dừng vì hết vòng là an toàn đã hoạt động |
| **Dừng giữa chừng** | Thế giới ở trạng thái dở dang — giữ `duong_di` để biết đã làm tới đâu |

**Bài tập:** `curriculum/09-agents/exercises/ex01_vong_lap_agent.py`

```powershell
pytest tests/phase09/test_ex01.py -v
```

➡️ Bài tiếp: **[02 — Bộ nhớ và ngữ cảnh](02_bo_nho_va_ngu_canh.ipynb)**